# Phase 2 - RMU unlearning: steering_coeff sweep (round 3)

Run in Colab with a GPU runtime (`Runtime` -> `Change runtime type` -> GPU).

Writes out the project's actual `src/common.py`, `src/unlearn.py`, `src/eval.py`, `configs/default.yaml` (kept in sync with the local repo), then trains+evaluates RMU on Qwen2.5-1.5B-Instruct across three candidate `steering_coeff` values (3, 4, 5) in one session, so we can compare trade-offs without a round trip per value.

**Why a sweep now:** round 1 (steering_coeff=6.5) gave real suppression (wmdp_bio 0.6787->0.3771) but also collapsed mmlu (0.6215->0.4669, way more than "a few points"). Round 2 (steering_coeff=2) had NO effect at all (forget_loss stayed flat, both scores unchanged). The useful trade-off is somewhere between 2 and 6.5 -- this sweep tests 3 intermediate values in one run.

**Known limitation, by owner's choice:** the official `cais/wmdp-bio-forget-corpus` is gated (HF access request required). This run substitutes an open PubMed abstract/article corpus (`ccdv/pubmed-summarization`) as the forget set instead -- same broad domain (biomedical literature) but not curated for the specific hazard-adjacent content the official corpus targets. Expect suppression to plausibly be weaker or less targeted than the published RMU results.

Safety reminder: this only trains the model to move ITS OWN internal activations away from a random fixed vector on forget-set text, and to stay close to its own frozen reference on retain-set text. No hazardous content is generated, read, or surfaced -- the forget-set text is only ever used as opaque input batches, never printed or inspected.

In [ ]:
!nvidia-smi
!pip install -q transformers==5.14.1 datasets==5.0.0 accelerate==1.14.0 peft==0.19.1 lm-eval==0.4.12

In [ ]:
import os
os.makedirs('src', exist_ok=True)
os.makedirs('configs', exist_ok=True)
os.makedirs('results', exist_ok=True)
os.makedirs('models', exist_ok=True)

In [ ]:
%%writefile src/common.py
"""Shared helpers: config loading, seeding, model loading, IO."""
import json
import os
import random
from pathlib import Path

import numpy as np
import torch
import yaml

REPO_ROOT = Path(__file__).resolve().parent.parent
DEFAULT_CONFIG_PATH = REPO_ROOT / "configs" / "default.yaml"


def load_config(config_path=None, overrides=None):
    """Load YAML config, deep-merge `overrides` (e.g. CLI flags) on top."""
    path = Path(config_path) if config_path else DEFAULT_CONFIG_PATH
    if not path.exists():
        raise FileNotFoundError(f"Config file not found: {path}")
    with open(path) as f:
        config = yaml.safe_load(f)
    if overrides:
        _deep_update(config, overrides)
    return config


def _deep_update(base, overrides):
    for key, value in overrides.items():
        if value is None:
            continue
        if isinstance(value, dict) and isinstance(base.get(key), dict):
            _deep_update(base[key], value)
        else:
            base[key] = value
    return base


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def resolve_device():
    return "cuda" if torch.cuda.is_available() else "cpu"


def resolve_dtype(name):
    return {"bfloat16": torch.bfloat16, "float16": torch.float16, "float32": torch.float32}[name]


def library_versions():
    import accelerate
    import datasets
    import transformers

    versions = {
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
        "accelerate": accelerate.__version__,
    }
    try:
        import lm_eval

        versions["lm_eval"] = lm_eval.__version__
    except ImportError:
        pass
    return versions


def load_model_and_tokenizer(model_id, dtype="bfloat16", device=None):
    from transformers import AutoModelForCausalLM, AutoTokenizer

    device = device or resolve_device()
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        dtype=resolve_dtype(dtype),
        device_map=device if device == "cuda" else None,
    )
    if device != "cuda":
        model = model.to(device)
    model.eval()
    return model, tokenizer


def save_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(data, f, indent=2)
    return path


def append_experiment_index(row, index_path=None):
    """Append a row (timestamp, phase, config hash, key metric) to the experiments index."""
    index_path = Path(index_path) if index_path else REPO_ROOT / "results" / "experiments_index.jsonl"
    index_path.parent.mkdir(parents=True, exist_ok=True)
    with open(index_path, "a") as f:
        f.write(json.dumps(row) + "\n")

In [ ]:
%%writefile src/eval.py
"""Run lm-eval on a model for the given tasks; save a normalized results JSON.

Usage:
    python -m src.eval --model <hf-id-or-path> --tasks wmdp_bio,mmlu --limit <int|None> --out results/<name>.json
"""
import argparse
import datetime
import sys

from src.common import library_versions, load_config, resolve_device, save_json, set_seed


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--model", required=True, help="HF model id or local path")
    p.add_argument("--tasks", required=True, help="Comma-separated lm-eval task names")
    p.add_argument("--limit", type=int, default=None, help="Sample limit per task (applies to all tasks given)")
    p.add_argument("--mmlu-limit", type=int, default=None, help="Override limit specifically for the mmlu task")
    p.add_argument("--out", required=True, help="Output JSON path")
    p.add_argument("--config", default=None, help="Path to YAML config (defaults to configs/default.yaml)")
    p.add_argument("--seed", type=int, default=None)
    p.add_argument("--batch-size", default="auto")
    return p.parse_args()


def extract_metric(task_results, preferred=("acc,none", "acc_norm,none", "acc", "acc_norm")):
    for key in preferred:
        if key in task_results:
            return key, task_results[key]
    # fall back to any key ending in a non-stderr accuracy-like metric
    for key, value in task_results.items():
        if not key.endswith("_stderr,none") and not key.endswith("_stderr") and "stderr" not in key:
            return key, value
    raise KeyError(f"No accuracy-like metric found in task results: {list(task_results.keys())}")


def extract_stderr(task_results, metric_key):
    base = metric_key.split(",")[0]
    suffix = "," + metric_key.split(",", 1)[1] if "," in metric_key else ""
    for candidate in (f"{base}_stderr{suffix}", f"{base}_stderr"):
        if candidate in task_results:
            return task_results[candidate]
    return None


def run_eval(model, tasks, limit=None, per_task_limit=None, seed=0, batch_size="auto"):
    import lm_eval
    from lm_eval.models.huggingface import HFLM

    per_task_limit = per_task_limit or {}
    lm = HFLM(pretrained=model, device=resolve_device(), batch_size=batch_size)

    results_out = {}
    for task in tasks:
        task_limit = per_task_limit.get(task, limit)
        raw = lm_eval.simple_evaluate(
            model=lm,
            tasks=[task],
            limit=task_limit,
            random_seed=seed,
            numpy_random_seed=seed,
            torch_random_seed=seed,
        )
        task_results = raw["results"][task]
        metric_key, acc = extract_metric(task_results)
        stderr = extract_stderr(task_results, metric_key)
        n = task_results.get("sample_count")
        if isinstance(n, dict):
            n = n.get(metric_key, next(iter(n.values()), None))
        if n is None:
            n = raw.get("n-samples", {}).get(task, {}).get("effective")
        results_out[task] = {"acc": acc, "acc_stderr": stderr, "n": n, "metric": metric_key}
    return results_out


def main():
    args = parse_args()
    config = load_config(args.config)
    seed = args.seed if args.seed is not None else config.get("seed", 0)
    set_seed(seed)

    tasks = [t.strip() for t in args.tasks.split(",") if t.strip()]
    per_task_limit = {}
    if args.mmlu_limit is not None and "mmlu" in tasks:
        per_task_limit["mmlu"] = args.mmlu_limit
    elif "mmlu" in tasks and args.limit is None:
        cfg_mmlu_limit = config.get("eval", {}).get("mmlu_limit")
        if cfg_mmlu_limit is not None:
            per_task_limit["mmlu"] = cfg_mmlu_limit

    print(f"Evaluating model={args.model} tasks={tasks} limit={args.limit} "
          f"per_task_limit={per_task_limit} seed={seed}", file=sys.stderr)

    results = run_eval(
        model=args.model,
        tasks=tasks,
        limit=args.limit,
        per_task_limit=per_task_limit,
        seed=seed,
        batch_size=args.batch_size,
    )

    import lm_eval

    output = {
        "model": args.model,
        "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "lm_eval_version": lm_eval.__version__,
        "tasks": results,
        "config": {
            "tasks": tasks,
            "limit": args.limit,
            "per_task_limit": per_task_limit,
            "seed": seed,
            "batch_size": args.batch_size,
            "library_versions": library_versions(),
        },
    }
    out_path = save_json(args.out, output)
    print(f"Wrote {out_path}")
    for task, r in results.items():
        print(f"  {task}: acc={r['acc']:.4f} (stderr={r['acc_stderr']}) n={r['n']} metric={r['metric']}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile src/unlearn.py
"""RMU (Representation Misdirection for Unlearning) adaptation.

Adapted from the reference implementation (Li et al. 2024, "The WMDP Benchmark"),
github.com/centerforaisafety/wmdp (rmu/unlearn.py, rmu/utils.py), verified against
the live repo source. Key differences from a naive port, all deliberate:
  - Target parameters are selected by NAME (e.g. "mlp.down_proj.weight"), not by a
    hardcoded positional index -- the reference's `param_ids=[6]` assumes a bias-less
    Mistral/Llama-style block; Qwen2 has attention biases, which shifts down_proj to
    index 9. Name-based selection is robust to that.
  - All non-target parameters are explicitly frozen (requires_grad=False), matching
    the project spec's "freeze all other parameters"; the reference relies on the
    optimizer's param group alone and never sets requires_grad=False elsewhere.
  - Forget-set substitute: the official cais/wmdp-bio-forget-corpus is gated
    (requires an HF access request). By owner's choice, this uses an open PubMed
    abstract/article corpus (ccdv/pubmed-summarization) instead -- topically in the
    same domain (biomedical literature) but not curated for hazard-adjacent content
    the way the official corpus is, so suppression may be weaker/less targeted. This
    is a known, documented limitation, not an attempt to reproduce the official corpus.
  - Dataset loading uses a bounded, non-streaming slice (`split[:N]`), NOT
    `streaming=True`: streaming proved unreliable against flaky network conditions
    (repeated range-request retries eventually crashed the Python interpreter
    itself, not just raised a catchable exception).
  - Retain dataset is `Salesforce/wikitext` (the bare `wikitext` repo id no longer
    resolves on the Hub -- it now requires the `Salesforce/` namespace).
  - The control vector's magnitude is CALIBRATED against the model's own measured
    activation scale at layer_id, not a raw absolute constant. A first real run using
    the reference's literal steering_coeff=6.5 (Zephyr-7B's tuned constant) produced
    a target vector ~4000x smaller than Qwen2.5-1.5B's actual activation magnitude at
    layer 6 -- forget_loss never moved across 150 steps and wmdp_bio was unchanged.
    steering_coeff is now a multiplier of the measured scale, making it portable across
    model families instead of tied to whatever model the reference happened to tune it on.

Usage:
    python -m src.unlearn --base-model Qwen/Qwen2.5-1.5B-Instruct \
        --layer-id 6 --layer-ids 4,5,6 --steering-coeff 6.5 --alpha 1200 \
        --lr 5e-5 --max-steps 150 --batch-size 4 --out models/unlearned
"""
import argparse
import datetime
import sys

import torch

from src.common import library_versions, load_config, resolve_device, resolve_dtype, save_json, set_seed

TARGET_PARAM_NAME = "mlp.down_proj.weight"


def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument("--base-model", default=None, help="HF model id or local path (default: config model.base)")
    p.add_argument("--layer-id", type=int, default=None, help="Layer whose activations are hooked for both losses")
    p.add_argument("--layer-ids", default=None, help="Comma-separated layer indices to update (e.g. 4,5,6)")
    p.add_argument("--steering-coeff", type=float, default=None)
    p.add_argument("--alpha", type=float, default=None)
    p.add_argument("--lr", type=float, default=None)
    p.add_argument("--max-steps", type=int, default=None)
    p.add_argument("--batch-size", type=int, default=None)
    p.add_argument("--max-seq-len", type=int, default=None, help="Truncation length for forget/retain batches")
    p.add_argument("--forget-dataset", default="ccdv/pubmed-summarization")
    p.add_argument("--forget-field", default="article")
    p.add_argument("--forget-split", default="train")
    p.add_argument("--forget-slice", type=int, default=3000, help="Rows to fetch before filtering/shuffling")
    p.add_argument("--retain-dataset", default="Salesforce/wikitext")
    p.add_argument("--retain-config", default="wikitext-2-raw-v1")
    p.add_argument("--retain-field", default="text")
    p.add_argument("--retain-split", default="test")
    p.add_argument("--retain-slice", type=int, default=3000, help="Rows to fetch before filtering/shuffling")
    p.add_argument("--seed", type=int, default=None)
    p.add_argument("--config", default=None)
    p.add_argument("--out", required=True)
    return p.parse_args()


def get_target_params(model, layer_ids, param_name=TARGET_PARAM_NAME):
    """Freeze every parameter, then return (unfrozen) the named param at each layer index."""
    for p in model.parameters():
        p.requires_grad_(False)
    selected = []
    for layer_id in layer_ids:
        layer = model.model.layers[layer_id]
        found = False
        for name, p in layer.named_parameters():
            if name == param_name:
                p.requires_grad_(True)
                selected.append(p)
                found = True
                break
        if not found:
            raise ValueError(f"Parameter '{param_name}' not found in layer {layer_id}")
    return selected


def forward_with_cache(model, inputs, module, no_grad=True):
    cache = {}

    def hook(_mod, _inp, out):
        cache["activation"] = out[0] if isinstance(out, tuple) else out

    handle = module.register_forward_hook(hook)
    try:
        if no_grad:
            with torch.no_grad():
                model(**inputs)
        else:
            model(**inputs)
    finally:
        handle.remove()
    return cache["activation"]


def load_text_iter(dataset_name, field, split, config=None, slice_n=3000, seed=0):
    """Non-streaming, bounded slice load. Deliberately NOT using `streaming=True`:
    it proved unreliable in practice (repeated range-request retries against flaky
    network conditions eventually crashed the Python interpreter itself, not just
    raised a catchable exception). A bounded slice download is slower to start but
    far more robust, and we only ever need a few thousand documents at most."""
    from datasets import load_dataset

    sliced_split = f"{split}[:{slice_n}]"
    ds = load_dataset(dataset_name, config, split=sliced_split) if config else \
        load_dataset(dataset_name, split=sliced_split)
    ds = ds.shuffle(seed=seed)
    for example in ds:
        text = example[field]
        if text and text.strip():
            yield text


def batched(iterator, batch_size):
    batch = []
    for item in iterator:
        batch.append(item)
        if len(batch) == batch_size:
            yield batch
            batch = []


def estimate_activation_scale(model, tokenizer, calibration_batches, module, max_seq_len, device):
    """Mean squared per-element activation magnitude at `module`, measured on real data.

    A raw random unit vector (L2 norm 1) scaled by a small constant like the reference
    implementation's steering_coeff=6.5 is utterly negligible next to real hidden-state
    magnitudes for some models -- e.g. on Qwen2.5-1.5B at layer 6, measured mean-squared
    activation was ~162, meaning a norm-6.5 target contributes ~0.03 to the per-element
    MSE (6.5^2 / hidden_size). The forget loss then can't move the model at all: gradient
    pressure toward a target ~4000x smaller than the activations themselves is negligible.
    This measures the real scale so steering_coeff can be a portable multiplier of it,
    instead of an absolute constant tuned for a different model family.
    """
    values = []
    with torch.no_grad():
        for batch in calibration_batches:
            inputs = tokenizer(
                batch, return_tensors="pt", padding=True, truncation=True, max_length=max_seq_len
            ).to(device)
            act = forward_with_cache(model, inputs, module, no_grad=True)
            values.append(act.pow(2).mean().item())
    return sum(values) / len(values) if values else 1.0


def run_rmu(
    model,
    frozen_model,
    tokenizer,
    forget_texts,
    retain_texts,
    layer_id,
    layer_ids,
    steering_coeff,
    alpha,
    lr,
    max_steps,
    batch_size,
    max_seq_len,
    device,
    n_calibration_batches=3,
):
    hidden_size = model.config.hidden_size
    module = model.model.layers[layer_id]
    frozen_module = frozen_model.model.layers[layer_id]

    forget_batches = batched(forget_texts, batch_size)
    retain_batches = batched(retain_texts, batch_size)

    calibration_batches = []
    for _ in range(n_calibration_batches):
        try:
            calibration_batches.append(next(forget_batches))
        except StopIteration:
            break
    typical_ms = estimate_activation_scale(model, tokenizer, calibration_batches, module, max_seq_len, device)
    typical_rms = typical_ms ** 0.5

    # unit_vector's per-element magnitude is ~1/sqrt(hidden_size); scaling by
    # steering_coeff * typical_rms * sqrt(hidden_size) makes each element's magnitude
    # ~steering_coeff * typical_rms -- i.e. steering_coeff natural-activation-scales.
    unit_vector = torch.rand(1, 1, hidden_size, device=device, dtype=model.dtype)
    unit_vector = unit_vector / unit_vector.norm()
    control_vec = unit_vector * (steering_coeff * typical_rms * (hidden_size ** 0.5))
    print(f"calibration: typical_ms={typical_ms:.4f} typical_rms={typical_rms:.4f} "
          f"control_vec_norm={control_vec.norm().item():.4f}", file=sys.stderr)

    params = get_target_params(model, layer_ids)
    optimizer = torch.optim.AdamW(params, lr=lr)

    history = []
    step = 0
    for forget_batch, retain_batch in zip(forget_batches, retain_batches):
        if step >= max_steps:
            break

        forget_inputs = tokenizer(
            forget_batch, return_tensors="pt", padding=True, truncation=True, max_length=max_seq_len
        ).to(device)
        retain_inputs = tokenizer(
            retain_batch, return_tensors="pt", padding=True, truncation=True, max_length=max_seq_len
        ).to(device)

        forget_activations = forward_with_cache(model, forget_inputs, module, no_grad=False)
        forget_loss = torch.nn.functional.mse_loss(forget_activations, control_vec.expand_as(forget_activations))

        retain_activations = forward_with_cache(model, retain_inputs, module, no_grad=False)
        with torch.no_grad():
            frozen_retain_activations = forward_with_cache(frozen_model, retain_inputs, frozen_module, no_grad=True)
        retain_loss = alpha * torch.nn.functional.mse_loss(retain_activations, frozen_retain_activations)

        loss = forget_loss + retain_loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        history.append({"step": step, "forget_loss": forget_loss.item(), "retain_loss": retain_loss.item()})
        print(f"step {step}: forget_loss={forget_loss.item():.4f} retain_loss={retain_loss.item():.4f}", file=sys.stderr)
        step += 1

    calibration = {
        "typical_ms": typical_ms,
        "typical_rms": typical_rms,
        "control_vec_norm": control_vec.norm().item(),
    }
    return history, calibration


def main():
    args = parse_args()
    config = load_config(args.config)
    seed = args.seed if args.seed is not None else config.get("seed", 0)
    set_seed(seed)

    rmu_cfg = config.get("rmu", {})
    base_model = args.base_model or config.get("model", {}).get("base")
    layer_id = args.layer_id if args.layer_id is not None else rmu_cfg.get("layer_id")
    if layer_id is None:
        raise ValueError("layer_id must be set via --layer-id or config rmu.layer_id")
    if args.layer_ids:
        layer_ids = [int(x) for x in args.layer_ids.split(",")]
    elif rmu_cfg.get("layer_ids"):
        layer_ids = list(rmu_cfg["layer_ids"])
    else:
        layer_ids = [layer_id - 2, layer_id - 1, layer_id]
    steering_coeff = args.steering_coeff if args.steering_coeff is not None else rmu_cfg.get("steering_coeff")
    alpha = args.alpha if args.alpha is not None else rmu_cfg.get("alpha")
    lr = args.lr if args.lr is not None else rmu_cfg.get("lr")
    max_steps = args.max_steps if args.max_steps is not None else rmu_cfg.get("max_steps")
    batch_size = args.batch_size if args.batch_size is not None else rmu_cfg.get("batch_size")
    max_seq_len = args.max_seq_len if args.max_seq_len is not None else config.get("model", {}).get("max_seq_len", 512)
    dtype_name = config.get("model", {}).get("dtype", "bfloat16")
    device = resolve_device()

    print(f"Loading base model {base_model} ...", file=sys.stderr)
    from transformers import AutoModelForCausalLM, AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(base_model)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(base_model, dtype=resolve_dtype(dtype_name))
    model = model.to(device)
    frozen_model = AutoModelForCausalLM.from_pretrained(base_model, dtype=resolve_dtype(dtype_name))
    frozen_model = frozen_model.to(device)
    frozen_model.eval()
    for p in frozen_model.parameters():
        p.requires_grad_(False)

    print(f"layer_id={layer_id} layer_ids={layer_ids} steering_coeff={steering_coeff} alpha={alpha} "
          f"lr={lr} max_steps={max_steps} batch_size={batch_size}", file=sys.stderr)

    forget_texts = load_text_iter(args.forget_dataset, args.forget_field, args.forget_split,
                                   slice_n=args.forget_slice, seed=seed)
    retain_texts = load_text_iter(args.retain_dataset, args.retain_field, args.retain_split,
                                   config=args.retain_config, slice_n=args.retain_slice, seed=seed)

    history, calibration = run_rmu(
        model=model,
        frozen_model=frozen_model,
        tokenizer=tokenizer,
        forget_texts=forget_texts,
        retain_texts=retain_texts,
        layer_id=layer_id,
        layer_ids=layer_ids,
        steering_coeff=steering_coeff,
        alpha=alpha,
        lr=lr,
        max_steps=max_steps,
        batch_size=batch_size,
        max_seq_len=max_seq_len,
        device=device,
    )

    model.save_pretrained(args.out)
    tokenizer.save_pretrained(args.out)

    meta = {
        "base_model": base_model,
        "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "config": {
            "layer_id": layer_id,
            "layer_ids": layer_ids,
            "steering_coeff": steering_coeff,
            "alpha": alpha,
            "lr": lr,
            "max_steps": max_steps,
            "batch_size": batch_size,
            "max_seq_len": max_seq_len,
            "seed": seed,
            "forget_dataset": args.forget_dataset,
            "forget_field": args.forget_field,
            "retain_dataset": args.retain_dataset,
            "retain_config": args.retain_config,
            "retain_field": args.retain_field,
            "library_versions": library_versions(),
        },
        "calibration": calibration,
        "history": history,
    }
    save_json(f"{args.out}/unlearn_run_meta.json", meta)
    print(f"Saved unlearned model to {args.out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile configs/default.yaml
model:
  base: "Qwen/Qwen2.5-1.5B-Instruct"
  fallback: "Qwen/Qwen2.5-3B-Instruct"
  dtype: "bfloat16"
  max_seq_len: 512

eval:
  tasks: ["wmdp_bio", "mmlu"]
  mmlu_limit: 500
  wmdp_bio_limit: null

seed: 0

rmu:
  layer_id: 6
  layer_ids: [4, 5, 6]
  steering_coeff: 2
  alpha: 1200
  lr: 5.0e-5
  max_steps: 150
  batch_size: 4

attack:
  data: "<benign general-domain dataset>"
  learning_rates: [1.0e-5, 5.0e-5]
  steps_grid: [0, 25, 50, 100, 200]
  batch_size: 4
  method: "full"

## Sweep steering_coeff over {3, 4, 5}

Each candidate trains a fresh unlearned model (150 steps, layer_id=6, alpha=1200 fixed) and immediately evaluates it on wmdp_bio + mmlu. The base model download is cached after the first candidate, so later candidates load faster. Watch each candidate's `calibration:` line and `forget_loss` trend as it runs.

In [ ]:
candidates = [3, 4, 5]

for c in candidates:
    print(f"\n{'='*20} steering_coeff={c} {'='*20}\n")
    out_dir = f"models/unlearned_c{c}"
    out_json = f"results/unlearned_c{c}.json"
    get_ipython().system(f"python -m src.unlearn --steering-coeff {c} --out {out_dir}")
    get_ipython().system(f"python -m src.eval --model {out_dir} --tasks wmdp_bio,mmlu --mmlu-limit 500 --batch-size auto --out {out_json}")

## Compare all candidates

In [ ]:
import json

baseline_wmdp, baseline_mmlu = 0.6787, 0.6215
print(f"{'candidate':<12}{'wmdp_bio':<12}{'mmlu':<12}")
print(f"{'baseline':<12}{baseline_wmdp:<12.4f}{baseline_mmlu:<12.4f}")
for c in candidates:
    d = json.load(open(f"results/unlearned_c{c}.json"))
    print(f"c={c:<10}{d['tasks']['wmdp_bio']['acc']:<12.4f}{d['tasks']['mmlu']['acc']:<12.4f}")

In [ ]:
# Print full JSONs for every candidate so they can be copied back to Claude Code.
for c in candidates:
    print(f"\n--- models/unlearned_c{c}/unlearn_run_meta.json ---")
    print(open(f"models/unlearned_c{c}/unlearn_run_meta.json").read())
    print(f"\n--- results/unlearned_c{c}.json ---")
    print(open(f"results/unlearned_c{c}.json").read())